# DEAP Emotion Recognition — Final Pipeline

**NeuroBarometer Project** | Апрель 2026

Мультимодальное распознавание эмоций на датасете DEAP (EEG + PPG + GSR).  
Модель: **LightMLP** с расширенным набором признаков.

| Модальность | Признаки | Описание |
|-------------|----------|----------|
| EEG (ch 0–31) | 160 | Differential Entropy × 5 полос × 32 канала |
| PPG (ch 38) | 10 | HRV: SDNN, RMSSD, pNN50, LF, HF, LF/HF и др. |
| GSR (ch 36) | 8 | EDA: SCL mean/std, SCR пики, slope |
| **Консенсус** | **2** | **Средняя оценка видео по остальным субъектам** |
| Позиция окна | 1 | Нормализованная позиция в trial (0→1) |
| FAA | 2 | Фронтальная альфа/тета асимметрия |
| Сегм. PPG/GSR | — | 3 × 20-сек сегмента вместо одного trial-уровня |

**Протоколы:** Subject-Dependent (SD, 8-fold CV) · Subject-Independent (LOSO)

## 0. Setup

In [ ]:
import sys, os
from pathlib import Path

PROJECT_DIR = Path(".").resolve()
sys.path.insert(0, str(PROJECT_DIR))
os.chdir(PROJECT_DIR)
print("Working dir:", PROJECT_DIR)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import json
import warnings
warnings.filterwarnings("ignore")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"PyTorch : {torch.__version__}")
print(f"Device  : {device}")
if device.type == "cuda":
    print(f"GPU     : {torch.cuda.get_device_name(0)}")

## 1. Запуск эксперимента

Запускает полный SD + LOSO и сохраняет JSON. Если результаты уже есть — можно пропустить и загрузить ниже.

In [ ]:
RUN_EXPERIMENT = False  # True = перезапустить (~30 мин SD + ~40 мин LOSO)

if RUN_EXPERIMENT:
    import subprocess
    result = subprocess.run(
        [sys.executable, "main.py", "--protocol", "both", "--model", "mlp", "--save-results"],
        capture_output=False, text=True
    )
else:
    print("Эксперимент пропущен. Загружаем готовые результаты ниже.")

## 2. Загрузка результатов

In [ ]:
from config import RESULTS_DIR

# Берём последний файл с результатами MLP both
result_files = sorted(RESULTS_DIR.glob("results_mlp_both_*.json"))
if not result_files:
    result_files = sorted(RESULTS_DIR.glob("results_*.json"))

result_path = result_files[-1]
print(f"Загружаем: {result_path.name}")

with open(result_path) as f:
    data = json.load(f)

sd   = data.get("sd_mlp") or data.get("sd")
loso = data.get("loso_mlp") or data.get("loso")

print(f"SD   aggregate: Val {sd['aggregate']['valence_acc'][0]:.2f}% | Ar {sd['aggregate']['arousal_acc'][0]:.2f}%")
if loso:
    print(f"LOSO aggregate: Val {loso['aggregate']['valence_acc'][0]:.2f}% | Ar {loso['aggregate']['arousal_acc'][0]:.2f}%")

## 3. Subject-Dependent (SD) — визуализация

In [ ]:
sids = sorted(sd["subjects"].keys(), key=int)
val_accs = [sd["subjects"][s]["mean"]["valence_acc"] for s in sids]
ar_accs  = [sd["subjects"][s]["mean"]["arousal_acc"]  for s in sids]

x = np.arange(len(sids))
w = 0.35

fig, ax = plt.subplots(figsize=(16, 5))
bars_v = ax.bar(x - w/2, val_accs, w, label="Valence",  color="#4472C4", alpha=0.85)
bars_a = ax.bar(x + w/2, ar_accs,  w, label="Arousal",  color="#ED7D31", alpha=0.85)

mean_v = np.mean(val_accs)
mean_a = np.mean(ar_accs)
ax.axhline(mean_v, color="#4472C4", linestyle="--", lw=1.8, label=f"Val mean {mean_v:.1f}%")
ax.axhline(mean_a, color="#ED7D31", linestyle="--", lw=1.8, label=f"Ar  mean {mean_a:.1f}%")
ax.axhline(50, color="gray", linestyle=":", lw=1.2, label="Chance 50%")

ax.set_xticks(x)
ax.set_xticklabels([f"s{int(s):02d}" for s in sids], rotation=45, fontsize=9)
ax.set_ylabel("Trial-level Accuracy (%)", fontsize=11)
ax.set_title("SD — Per-subject accuracy (8-fold CV, majority vote)", fontsize=13)
ax.legend(fontsize=10)
ax.set_ylim(40, 105)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("results/sd_per_subject.png", dpi=150, bbox_inches="tight")
plt.show()
print(f"SD Valence: {mean_v:.2f}% +/- {np.std(val_accs):.2f}%")
print(f"SD Arousal: {mean_a:.2f}% +/- {np.std(ar_accs):.2f}%")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, accs, name, color in zip(axes,
                                  [val_accs, ar_accs],
                                  ["Valence", "Arousal"],
                                  ["#4472C4", "#ED7D31"]):
    ax.hist(accs, bins=10, color=color, alpha=0.8, edgecolor="white")
    ax.axvline(np.mean(accs), color="black", linestyle="--", lw=2,
               label=f"Mean {np.mean(accs):.1f}%")
    ax.set_xlabel("Accuracy (%)", fontsize=11)
    ax.set_ylabel("Subjects", fontsize=11)
    ax.set_title(f"SD {name} — distribution across subjects", fontsize=12)
    ax.legend(fontsize=10)
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig("results/sd_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 4. Subject-Independent (LOSO) — визуализация

In [ ]:
if loso is None:
    print("LOSO результаты не найдены в файле.")
else:
    sids_l  = sorted(loso["subjects"].keys(), key=int)
    val_l   = [loso["subjects"][s]["valence_acc"] for s in sids_l]
    ar_l    = [loso["subjects"][s]["arousal_acc"]  for s in sids_l]

    x = np.arange(len(sids_l))
    fig, ax = plt.subplots(figsize=(16, 5))
    ax.bar(x - w/2, val_l, w, label="Valence", color="#4472C4", alpha=0.85)
    ax.bar(x + w/2, ar_l,  w, label="Arousal", color="#ED7D31", alpha=0.85)
    ax.axhline(np.mean(val_l), color="#4472C4", linestyle="--", lw=1.8,
               label=f"Val mean {np.mean(val_l):.1f}%")
    ax.axhline(np.mean(ar_l), color="#ED7D31", linestyle="--", lw=1.8,
               label=f"Ar  mean {np.mean(ar_l):.1f}%")
    ax.axhline(50, color="gray", linestyle=":", lw=1.2, label="Chance 50%")
    ax.set_xticks(x)
    ax.set_xticklabels([f"s{int(s):02d}" for s in sids_l], rotation=45, fontsize=9)
    ax.set_ylabel("Trial-level Accuracy (%)", fontsize=11)
    ax.set_title("LOSO — Per-subject accuracy (leave-one-subject-out)", fontsize=13)
    ax.legend(fontsize=10)
    ax.set_ylim(30, 105)
    ax.grid(axis="y", alpha=0.3)
    plt.tight_layout()
    plt.savefig("results/loso_per_subject.png", dpi=150, bbox_inches="tight")
    plt.show()
    print(f"LOSO Valence: {np.mean(val_l):.2f}% +/- {np.std(val_l):.2f}%")
    print(f"LOSO Arousal: {np.mean(ar_l):.2f}% +/- {np.std(ar_l):.2f}%")

## 5. Ablation Study — вклад каждой гипотезы

Пошаговое добавление признаков, начиная с базового LightMLP (только DE + HRV + EDA).

In [ ]:
import pandas as pd

ablation = [
    {"Конфигурация": "Базовый LightMLP (DE + HRV + EDA)",            "Val %": 66.6, "Ar %": 64.2, "Delta Val": "+0.0", "Статус": "Baseline"},
    {"Конфигурация": "+ H1: Групповой консенсус по видео",            "Val %": 79.6, "Ar %": 70.9, "Delta Val": "+13.0", "Статус": "Принято"},
    {"Конфигурация": "+ H2: Baseline DE нормализация",                "Val %": 78.2, "Ar %": 69.2, "Delta Val": "-1.4",  "Статус": "Откат"},
    {"Конфигурация": "+ H3: Позиция окна в trial",                   "Val %": 79.9, "Ar %": 70.1, "Delta Val": "+0.3",  "Статус": "Нейтрально"},
    {"Конфигурация": "+ H4: Фронтальная альфа/тета асимметрия (FAA)", "Val %": 79.7, "Ar %": 70.4, "Delta Val": "-0.2",  "Статус": "Нейтрально"},
    {"Конфигурация": "+ H5: Сегментированные PPG/GSR (3 x 20 с)",    "Val %": 82.2, "Ar %": 71.8, "Delta Val": "+2.3",  "Статус": "Принято"},
]

df_abl = pd.DataFrame(ablation)
df_abl.set_index("Конфигурация", inplace=True)
print(df_abl.to_string())

In [ ]:
configs = [
    "Baseline",
    "+H1 Consensus",
    "+H2 Baseline DE",
    "+H3 Position",
    "+H4 FAA",
    "+H5 Seg. PPG/GSR",
]
val_abl = [66.6, 79.6, 78.2, 79.9, 79.7, 82.2]
ar_abl  = [64.2, 70.9, 69.2, 70.1, 70.4, 71.8]

x = np.arange(len(configs))
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(x, val_abl, "o-", color="#4472C4", lw=2.2, ms=8, label="Valence")
ax.plot(x, ar_abl,  "s--", color="#ED7D31", lw=2.2, ms=8, label="Arousal")

for i, (v, a) in enumerate(zip(val_abl, ar_abl)):
    ax.annotate(f"{v:.1f}", (i, v), textcoords="offset points", xytext=(0, 10),
                ha="center", fontsize=9, color="#4472C4")
    ax.annotate(f"{a:.1f}", (i, a), textcoords="offset points", xytext=(0, -16),
                ha="center", fontsize=9, color="#ED7D31")

ax.axhline(50, color="gray", linestyle=":", lw=1.2, label="Chance")
ax.set_xticks(x)
ax.set_xticklabels(configs, rotation=20, ha="right", fontsize=10)
ax.set_ylabel("SD Accuracy (%)", fontsize=11)
ax.set_title("Ablation Study — накопительный эффект признаков", fontsize=13)
ax.legend(fontsize=10)
ax.set_ylim(55, 90)
ax.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig("results/ablation.png", dpi=150, bbox_inches="tight")
plt.show()

## 6. Итоговая таблица результатов

In [ ]:
sd_agg   = sd["aggregate"]
loso_agg = loso["aggregate"] if loso else None

rows = [
    {
        "Модель / Протокол":  "LightMLP + Features  [SD]",
        "Valence Acc":  f"{sd_agg['valence_acc'][0]:.2f} +/- {sd_agg['valence_acc'][1]:.2f}%",
        "Arousal Acc":  f"{sd_agg['arousal_acc'][0]:.2f} +/- {sd_agg['arousal_acc'][1]:.2f}%",
        "Valence F1":   f"{sd_agg['valence_f1'][0]:.2f}%",
        "Arousal F1":   f"{sd_agg['arousal_f1'][0]:.2f}%",
    },
]
if loso_agg:
    rows.append({
        "Модель / Протокол":  "LightMLP + Features  [LOSO]",
        "Valence Acc":  f"{loso_agg['valence_acc'][0]:.2f} +/- {loso_agg['valence_acc'][1]:.2f}%",
        "Arousal Acc":  f"{loso_agg['arousal_acc'][0]:.2f} +/- {loso_agg['arousal_acc'][1]:.2f}%",
        "Valence F1":   f"{loso_agg['valence_f1'][0]:.2f}%",
        "Arousal F1":   f"{loso_agg['arousal_f1'][0]:.2f}%",
    })

# Базовые результаты для сравнения
rows.extend([
    {"Модель / Протокол": "--- Сравнение (литература) ---",
     "Valence Acc": "", "Arousal Acc": "", "Valence F1": "", "Arousal F1": ""},
    {"Модель / Протокол": "Базовый LightMLP  [SD]",
     "Valence Acc": "66.6%", "Arousal Acc": "64.2%",
     "Valence F1": "—", "Arousal F1": "—"},
    {"Модель / Протокол": "EEGNet multimodal  [SD]",
     "Valence Acc": "65.3%", "Arousal Acc": "63.0%",
     "Valence F1": "—", "Arousal F1": "—"},
    {"Модель / Протокол": "SOTA статьи DEAP (SD, с утечкой)",
     "Valence Acc": "88–99%", "Arousal Acc": "87–99%",
     "Valence F1": "—", "Arousal F1": "—"},
])

df_res = pd.DataFrame(rows).set_index("Модель / Протокол")
df_res

## 7. Ключевые выводы

### Что работает
1. **Групповой консенсус по видео** (+13% Valence) — самый мощный признак.  
   Физический смысл: все 32 субъекта смотрели одни и те же 40 видео. Среднее мнение группы о видео — сильный prior на его эмоциональную нагрузку.

2. **Сегментированные PPG/GSR** (+2.3% Valence) — деление trial на 3 сегмента × 20 сек вместо единого вектора даёт модели темпоральную информацию о вегетативной динамике.

### Что не помогло
- **Baseline DE нормализация**: DE уже нормализуется StandardScaler-ом в тренировочном цикле.
- **Позиция окна / FAA**: нейтральны — модель уже учится этому из DE признаков.

### Почему SOTA статьи дают 90%+
Большинство SOTA результатов на DEAP содержат **data leakage**:
- Окна одного trial попадают в train и test одновременно
- StandardScaler fit на всём датасете (включая test)
- Фиксированный порог 5.0 → дисбаланс классов → тривиальное предсказание мажоритарного класса

Наш пайплайн реализует **честный** протокол: GroupKFold по trials, scaler fit только на train, медианный порог per-subject.

### Перенос на NeuroBarometer
Признак «консенсус по видео» **недоступен в реальном времени** (нет других участников, смотрящих то же видео).  
Для собственного датасета NeuroBarometer целевой показатель: **~66–70% SD** без консенсуса, **~75%+ SD** при накоплении данных нескольких пользователей.